<a href="https://colab.research.google.com/github/E-HAZMATs/FT-SmolLM/blob/main/FT_SmolLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip -qqq install transformers datasets trl torch
! pip -qqq install peft

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
dataset_name = 'HuggingFaceTB/smoltalk2'
ds = load_dataset(dataset_name, 'SFT', streaming=True)

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

In [ ]:
ds.keys()

dict_keys(['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_everyday_convs_reasoning_Qwen3_32B_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_think', 'xlam_traces_no_th

In [ ]:
split = 'Mixture_of_Thoughts_science_no_think'

In [ ]:
model_name = "HuggingFaceTB/SmolLM-135M"
base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name + '-Instruct')

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
instucut_name = model_name + '-Instruct'
instruct = AutoModelForCausalLM.from_pretrained(instucut_name).to(device)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
print(f"Model Size: {(base.get_memory_footprint() / 1024**3):.3f}GB")

Model Size: 0.251GB


In [ ]:
from datasets import Dataset
examples_list = list(ds[split].take(5))
examples = Dataset.from_list(examples_list)

In [ ]:
tokenizer.chat_template

"{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [ ]:
def format_ds(batch):
    return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}

text = examples.map(format_ds, batched=True, remove_columns=examples.column_names)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
text[0]

{'text': "<|im_start|>user\nWhat hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol<|im_end|>\n<|im_start|>assistant\nCaffeine acts as a diuretic by inhibiting the action of antidiuretic hormone (ADH), which is responsible for signaling the kidneys to reabsorb water and concentrate urine. When ADH is suppressed, the kidneys excrete more water, leading to increased urination. The other hormones listed—insulin (regulates blood sugar), thyroxine (regulates metabolism), and cortisol (involved in stress response)—are not directly related to fluid balance or diuresis. \n\n**Answer: A**  \n\\boxed{A}<|im_end|>\n"}

In [ ]:
tokenizer.eos_token_id

2

In [ ]:
prompt = "I'd like a recipe for something warm."
tokenized = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
  outputs = base.generate(
      **tokenized,
      max_new_tokens=200,
      temperature=0.7,
      do_sample=True,
      pad_token_id=tokenizer.eos_token_id
  )
  decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
  print(decoded)



I'd like a recipe for something warm.

I read the same post in the comments and thought I would share my own experience of how I made a little lamb stew.

I've always taken lamb well to heart, though I have always had difficulty cooking it well. It is a little bit different in the UK, especially on Sundays. I tend to eat at home and I eat too much lamb.

I started cooking lamb on Sunday, so I felt like I was ready to start eating again. That is when I decided to make this lamb stew.

The lamb that I cooked this week was called lamb chops. I had no idea what lamb was. I had never heard of it before. I had never even heard of lamb before. I had never had it. I had never even tried it. I was going to make some lamb stew but I couldn't because I wasn't sure what to do with it. I had to eat some lamb before deciding what to make. And I had to eat


In [ ]:
torch.cuda.is_available()

True

In [ ]:
base.device

device(type='cuda', index=0)

In [ ]:
instruct.name_or_path

'HuggingFaceTB/SmolLM-135M-Instruct'

In [ ]:
# Sampling from the instruct model

message = [{
    "role": "user",
    "content": prompt
}]

formatted = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

formatted_tokenized = tokenizer(formatted, return_tensors='pt').to(device)

output = instruct.generate(
    **formatted_tokenized,
    max_new_tokens=200,
    temperature=0.5,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

decoded = tokenizer.decode(output[0])
print(decoded)
#

<|im_start|>user
I'd like a recipe for something warm.<|im_end|>
<|im_start|>assistant
What a wonderful idea! Here's a recipe for a warm and comforting warm breakfast:

**Ingredients:**

For the eggs:

* 2 large eggs
* 1/2 cup milk
* 1/4 cup granulated sugar
* 1/4 teaspoon salt

For the flour:

* 2 cups all-purpose flour
* 1 tablespoon baking powder
* 1 teaspoon salt

For the butter:

* 1/2 cup unsalted butter, softened
* 1/4 cup unsalted butter, softened
* 1/2 cup milk
* 1 teaspoon vanilla extract

For the jam or preserves:

* 1/2 cup powdered sugar
* 1/2 cup granulated sugar
* 1/4 teaspoon salt

For the whipped cream:

* 1 cup heavy cream
* 1/2 cup granulated sugar
* 


In [ ]:
assistant_start = tokenizer.bos_token + 'assistant\n'
response_start = decoded.find(assistant_start) + len(assistant_start)
response = decoded[response_start:].split(tokenizer.eos_token)[0]
print(response)

What a wonderful idea! Here's a recipe for a warm and comforting warm breakfast:

**Ingredients:**

For the eggs:

* 2 large eggs
* 1/2 cup milk
* 1/4 cup granulated sugar
* 1/4 teaspoon salt

For the flour:

* 2 cups all-purpose flour
* 1 tablespoon baking powder
* 1 teaspoon salt

For the butter:

* 1/2 cup unsalted butter, softened
* 1/4 cup unsalted butter, softened
* 1/2 cup milk
* 1 teaspoon vanilla extract

For the jam or preserves:

* 1/2 cup powdered sugar
* 1/2 cup granulated sugar
* 1/4 teaspoon salt

For the whipped cream:

* 1 cup heavy cream
* 1/2 cup granulated sugar
* 


### Now FT the base model

In [ ]:
def preprocess(batch):
  return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}


In [ ]:
training_config = SFTConfig(
    # Model and data
    output_dir=f"./{model_name}",
    dataset_text_field="text",
    max_length=512,

    # Training hyperparameters
    per_device_train_batch_size=4,  # Adjust based on your GPU memory
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=1,  # Start with 1 epoch
    max_steps=20,  # Limit steps for demo

    # Optimization
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # Logging and saving
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,

    # Memory optimization
    dataloader_num_workers=0,
    # group_by_length=True,  # Group similar length sequences

    # Hugging Face Hub integration
    push_to_hub=False,  # Set to True to upload to Hub
    hub_model_id=f"your-username/{model_name}",

    # Experiment tracking
    # report_to=["trackio"],  # Use trackio for experiment tracking
    run_name=f"{model_name}-training",
)

print("Training configuration set!")
print(f"Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

Training configuration set!
Effective batch size: 8


In [ ]:
from datasets import interleave_datasets
used_splits = [
    'smoltalk_smollm3_everyday_conversations_no_think','Mixture_of_Thoughts_science_no_think',
    'tulu_3_sft_personas_instruction_following_no_think',
    'smoltalk_multilingual_8languages_lang_5_no_think',
    'table_gpt_no_think'
    ]

selected_ds = interleave_datasets([ds[split] for split in used_splits])
selected_ds = selected_ds.map(preprocess, batched=True, remove_columns=selected_ds.column_names)

In [ ]:
trainer = SFTTrainer(
    model=base,
    train_dataset=selected_ds,
    args=training_config
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,2.423009
20,2.328940


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=2.3759742736816407, metrics={'train_runtime': 65.0252, 'train_samples_per_second': 2.461, 'train_steps_per_second': 0.308, 'total_flos': 50653950746112.0, 'train_loss': 2.3759742736816407, 'epoch': 1.0})